In [ ]:
# Merge đơn giản: lấy cột của file CSV đầu tiên, gộp tất cả file trong ../data/preprocessed
# Ghi ra ../data/unified/preprocessed_merged.csv

from pathlib import Path
import pandas as pd
import re

IN_DIR  = Path("../data/preprocessed")
OUT_CSV = Path("../data/unified/preprocessed_merged.csv") 

IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

def clean_and_dedup_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Loại BOM, trim khoảng trắng, gom cột trùng (giữ cột đầu tiên)
    new_cols = []
    keep_idx = []
    seen = set()
    for idx, c in enumerate(df.columns):
        col = str(c).replace("\ufeff", "")       # bỏ BOM
        col = re.sub(r"\s+", " ", col).strip()   # gọn khoảng trắng
        key = col.lower()                        # so trùng không phân biệt hoa/thường
        if key not in seen:
            seen.add(key)
            new_cols.append(col)
            keep_idx.append(idx)
        else:
            # bỏ cột trùng, giữ bản đầu tiên
            continue
    df = df.iloc[:, keep_idx]
    df.columns = new_cols
    return df

def read_relaxed_csv(p: Path) -> pd.DataFrame:
    # Đọc “khoan dung”: ưu tiên utf-8-sig, bỏ qua byte lỗi để tránh UnicodeDecodeError
    return pd.read_csv(
        p,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
        encoding_errors="ignore",
        on_bad_lines="skip"
    )

paths = sorted(IN_DIR.glob("*.csv"))
if not paths:
    print(f"Không tìm thấy CSV trong {IN_DIR.resolve()}")
else:
    # File đầu tiên xác định bộ cột chuẩn
    df0 = read_relaxed_csv(paths[0])
    df0 = clean_and_dedup_columns(df0)
    base_cols = list(df0.columns)
    frames = [df0[base_cols]]
    print(f"Chuẩn cột dựa trên: {paths[0].name} -> {base_cols}")

    # Các file còn lại: chỉ lấy đúng các cột base_cols (thiếu thì bù cột rỗng)
    for p in paths[1:]:
        dfi = read_relaxed_csv(p)
        dfi = clean_and_dedup_columns(dfi)
        for c in base_cols:
            if c not in dfi.columns:
                dfi[c] = ""
        frames.append(dfi[base_cols])
        print(f"Đã gộp: {p.name} (rows={len(dfi)})")

    merged = pd.concat(frames, ignore_index=True)
    merged.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Gộp xong {len(paths)} file → {OUT_CSV} ({len(merged)} dòng)")
    try:
        display(merged.head())
    except:
        pass


Chuẩn cột dựa trên: batdongsan_preprocessed.csv -> ['tieu_de', 'gia', 'dia_chi', 'dien_tich_dat', 'phong_ngu', 'phong_tam', 'so_tang', 'phap_ly', 'ngay_dang']
Đã gộp: mogi_preprocessed.csv (rows=2079)
Đã gộp: muaban_preprocessed.csv (rows=1759)
Đã gộp: thuviennhadat_preprocessed.csv (rows=1567)
Gộp xong 4 file → ..\data\unified\preprocessed_merged.csv (7006 dòng)


,tieu_de,gia,dia_chi,dien_tich_dat,phong_ngu,phong_tam,so_tang,phap_ly,ngay_dang
0,Chính chủ nhờ đăng tin bán nhà hẻm xe hơi 94/8...,21.5,"94/8 Đường Trần Khắc Chân, Phường Tân Định, Qu...",32.0,3,3,3,Sổ hồng,2025-11-22
1,"Nhà 2 mặt tiền 15 Phan Tôn, P. Tân Định Quận 1",16.5,"15, Đường Phan Tôn, Phường Đa Kao, Quận 1, Hồ ...",55.0,3,2,2,Sổ hồng,2025-11-26
2,"Bán nhà trung tâm q1 hẻm 457 Nguyễn Cảnh Chân,...",17.5,"Đường Nguyễn Cảnh Chân, Phường Cầu Kho, Quận 1...",66.0,5,6,4,Sổ hồng,2025-11-21
3,"Bán gấp giảm nhanh 1,5 tỷ còn 16 tỷ nhà Nguyễn...",16.0,"Đường Nguyễn Thái Bình, Phường Nguyễn Thái Bìn...",73.0,8,8,5,Sổ hồng,2025-11-19
4,THẬT 100% THU 250 TRIỆU/THÁNG 46 PHÒNG 10mx17m...,49.9,"Đường Bùi Thị Xuân, Phường Bến Thành, Quận 1, ...",160.0,46,46,8,Sổ hồng,2025-11-23
